# 04. Construção da Base de Conhecimento Vetorial (ChromaDB & Embeddings)

**Contexto:** Tech Challenge 04 - Voice of Customer Intelligence com RAG  
**Requisitos do Desafio:** Criação de Embeddings, Armazenamento, Indexação e Exploração Contextual com Metadados.  

Nesta etapa, conectamos o corpus textual enriquecido e pré-processado (`reviews_enriched_for_rag.csv`) a um **Banco de Dados Vetorial (ChromaDB)**. Cada avaliação e seus metadados de compra/categoria são convertidos em representações vetoriais densas (*Embeddings*) e indexados com métrica de similaridade por **cosseno**.

In [1]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
import os
from tqdm import tqdm

DATA_DIR = '../data/'
PROCESSED_DIR = '../data/processed/'
DB_PATH = '../data/vector_db/'

# Carregamento do dataset pré-processado e consolidado
df_docs = pd.read_csv(f'{PROCESSED_DIR}reviews_enriched_for_rag.csv')
print(f"Total de documentos para indexação: {len(df_docs):,}")

Total de documentos para indexação: 40,668


## 1. Configuração do ChromaDB e Modelo de Embeddings

**Decisão Arquitetural e de Custo:**  
Para indexar mais de 40.000 avaliações sem incorrer em custos proibitivos de API externa ou limites de taxa (rate limits), utilizamos a função `DefaultEmbeddingFunction()` do ChromaDB (`all-MiniLM-L6-v2` via `sentence-transformers`).  
- Execução 100% local e eficiente em CPU/GPU.
- Espaço métrico configurado com **similaridade de cosseno** (`hnsw:space: cosine`).

In [2]:
# Inicialização do cliente persistente do ChromaDB
client = chromadb.PersistentClient(path=DB_PATH)
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

collection_name = "olist_reviews"

# A coleção persistida anteriormente pode ter sido criada com outra embedding function
# (ex.: 'default') ou sem os metadados estruturados. Reabri-la com o modelo multilíngue
# gera conflito de EF no ChromaDB; por isso removemos qualquer coleção pré-existente para
# garantir um espaço vetorial consistente antes da reindexação.
try:
    client.delete_collection(collection_name)
    print(f"Coleção '{collection_name}' pré-existente removida para reindexação limpa.")
except Exception:
    pass

collection = client.create_collection(
    name=collection_name,
    embedding_function=sentence_transformer_ef,
    metadata={"hnsw:space": "cosine"}
)

print(f"Coleção '{collection_name}' pronta. Registros atuais: {collection.count():,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Coleção 'olist_reviews' pré-existente removida para reindexação limpa.
Coleção 'olist_reviews' pronta. Registros atuais: 0


## 2. Indexação dos Documentos em Lotes (Batches)
Para garantir eficiência de memória e tolerância a falhas, a ingestão no ChromaDB é realizada em lotes de 1.000 documentos. Além do texto, cada vetor é indexado com um conjunto de **metadados estruturados** (`review_score`, `categoria`, `ano`, `mes`, `ano_mes`) que habilita a **filtragem contextual** real na recuperação (Desafio Adicional).

In [3]:
batch_size = 1000
total_docs = len(df_docs)

# O esquema de metadados foi ampliado (categoria, ano, mes, ano_mes). Recriamos a coleção
# sempre que ela estiver ausente, incompleta ou sem os novos metadados, garantindo
# reindexação limpa e consistente com os filtros expostos no pipeline RAG.
EXPECTED_METADATA_KEYS = {"review_score", "categoria", "ano", "mes", "ano_mes"}

def _metadados_completos(col):
    if col.count() == 0:
        return False
    amostra = col.get(limit=1, include=["metadatas"])
    if not amostra["metadatas"]:
        return False
    return EXPECTED_METADATA_KEYS.issubset(set(amostra["metadatas"][0].keys()))

if collection.count() == total_docs and _metadados_completos(collection):
    print(f"A base já contém {collection.count():,} registros com metadados completos. Etapa concluída.")
else:
    print("Recriando a coleção para reindexação com metadados estruturados...")
    client.delete_collection(collection_name)
    collection = client.create_collection(
        name=collection_name,
        embedding_function=sentence_transformer_ef,
        metadata={"hnsw:space": "cosine"}
    )

    for i in tqdm(range(0, total_docs, batch_size)):
        batch = df_docs.iloc[i:i+batch_size]

        documents = batch['rag_document'].tolist()
        ids = batch['review_id'].tolist()
        metadatas = [
            {
                "review_score": int(row.review_score),
                "categoria":    str(row.categoria),
                "ano":          int(row.ano),
                "mes":          int(row.mes),
                "ano_mes":      int(row.ano_mes),
            }
            for row in batch.itertuples(index=False)
        ]

        collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
    print(f"\nIndexação concluída! Total indexado: {collection.count():,} registros.")

Recriando a coleção para reindexação com metadados estruturados...



  0%|          | 0/41 [00:00<?, ?it/s]


  2%|▏         | 1/41 [00:04<03:12,  4.81s/it]


  5%|▍         | 2/41 [00:09<03:03,  4.70s/it]


  7%|▋         | 3/41 [00:13<02:55,  4.61s/it]

...


 95%|█████████▌| 39/41 [02:59<00:09,  4.53s/it]


 98%|█████████▊| 40/41 [03:03<00:04,  4.48s/it]


100%|██████████| 41/41 [03:06<00:00,  4.00s/it]


100%|██████████| 41/41 [03:06<00:00,  4.55s/it]


Indexação concluída! Total indexado: 40,668 registros.


## 3. Teste de Recuperação com Filtragem de Metadados (Desafio Adicional)
Validamos a recuperação vetorial combinando múltiplos filtros de metadado. O ChromaDB exige o operador `$and` para combinar condições; abaixo filtramos por **nota baixa** (`review_score <= 2`) **E** **período** (`ano == 2018`) para isolar queixas recentes e eliminar falsos positivos semânticos.

In [4]:
query_teste = "Atraso na entrega e suporte ruim"

# Filtro combinado de metadados estruturados (nota + período). O ChromaDB
# exige $and para combinar mais de uma condição no operador where.
resultados = collection.query(
    query_texts=[query_teste],
    n_results=3,
    where={"$and": [
        {"review_score": {"$lte": 2}},
        {"ano": {"$eq": 2018}},
    ]}
)

print(f"Consulta realizada: '{query_teste}'")
print("Filtro aplicado: review_score <= 2 AND ano == 2018")
print("\n=== Evidências Recuperadas ===")
for i, (doc, doc_id, meta) in enumerate(zip(resultados['documents'][0], resultados['ids'][0], resultados['metadatas'][0]), 1):
    print(f"[{i}] ID: {doc_id} | Metadados: {meta}")
    print(f"    {doc}")
    print("-"*60)

Consulta realizada: 'Atraso na entrega e suporte ruim'
Filtro aplicado: review_score <= 2 AND ano == 2018

=== Evidências Recuperadas ===
[1] ID: f9da62bb76ddba27efd3a9111e2040a2 | Metadados: {'ano_mes': 201801, 'categoria': 'telefonia', 'ano': 2018, 'mes': 1, 'review_score': 1}
    Categoria: telefonia | Nota: 1 | Data: 2018-01-04 | Comentário: o unico ponto positivo foi a entrega dentro do prazo, produto de péssima qualidade, parte interna solta da carcaça e para piorar não funciona. foi dinheiro jogado fora. não recomendo a ninguém.
------------------------------------------------------------
[2] ID: 4baf65ba0f4dc753e9cf3d8863433da0 | Metadados: {'review_score': 2, 'ano': 2018, 'categoria': 'papelaria', 'mes': 1, 'ano_mes': 201801}
    Categoria: papelaria | Nota: 2 | Data: 2018-01-29 | Comentário: falta de consideração com o cliente, pois ocorreu problema na entrega é em momento nenhum o problema foi tratado.duas compras, uma com prazo de entrega e , as atrasou, liguei dia reclaman